In [2]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)
print("device count:", torch.cuda.device_count())

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


torch: 2.10.0+cu128
cuda available: False
torch cuda version: 12.8
device count: 0


/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [3]:
!nvidia-smi

Failed to initialize NVML: Unknown Error


In [3]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()  # 선택

In [4]:
import os
print(os.environ.get("CUDA_VISIBLE_DEVICES"))

None


In [ ]:
# ablate_runner.py
# ============================================================
# [이 코드가 뭐 하는 코드인지 쉽게 설명]
#
# 이 코드는 "부분적으로만 관측되는 CartPole 환경"에서
# LSTM Actor-Critic 모델을 학습시키고,
# "좋은 성능이 나오는 학습 조건"을 찾기 위한 실험 코드다.
#
# 쉽게 말하면:
# 1) 기본 RL 모델(Actor-Critic + LSTM)을 만든다.
# 2) 학습 중 성능이 갑자기 떨어지면(collapse), 정책을 너무 흔들리지 않게 안정화한다.
# 3) 학습 중 성능이 갑자기 좋아지면(spike), 그 좋은 행동을 더 따라가게 만든다.
# 4) 이런 장치들의 강도(extra update, KL, SIL 등)를 바꿔가며
#    "어떤 설정이 제일 잘 학습되는지" 비교한다.
#
# ------------------------------------------------------------
# [좋은 모델을 어떻게 튜닝하는 코드인가?]
#
# 이 코드는 단순히 한 번 학습만 하는 게 아니라,
# "학습을 더 잘 되게 만드는 하이퍼파라미터 / 전략"을 비교하는 코드다.
#
# 튜닝 아이디어는 크게 3가지:
#
# (1) extra_updates_on_event
#     - 성능에 의미 있는 변화(collapse/spike)가 생겼을 때
#       actor를 몇 번 더 업데이트할지 정함
#     - 0,1,2,4,6처럼 바꿔가며 어떤 강도가 좋은지 실험
#
# (2) Phase Schedule (탐색 -> 수렴)
#     - 초반에는 많이 탐색하도록 entropy를 크게 유지
#     - 후반에는 탐색을 줄이고 안정적으로 수렴하도록 조정
#     - SIL은 후반에 줄이고, KL anchor는 후반에 강화
#
# (3) Ablation
#     - KL anchor 제거, SIL 제거, event update 제거 등
#       각 구성요소를 하나씩 빼 보면서
#       "어떤 장치가 실제로 성능에 기여하는지" 확인
#
# 즉, 이 코드는
# "좋은 RL 모델을 만드는 코드"이면서 동시에
# "좋은 학습 전략을 고르는 튜닝 실험 코드"다.
#
# ------------------------------------------------------------
# [서버에서 오래 돌릴 때 중요한 점]
#
# 이 코드에 체크포인트(resume)가 들어가 있어도,
# "컴퓨터를 꺼도 서버에서 계속 도는 것"은 실행 방식이 따로 중요하다.
#
# 예:
#   nohup python ablate_runner.py --device cuda:0 --resume > train.log 2>&1 &
#
# 또는 tmux / screen 사용.
#
# 정리:
# - 서버에서 계속 돌아가게 하는 것  -> nohup / tmux / screen
# - 중간에 끊겨도 이어서 하는 것   -> 이 코드의 checkpoint + --resume
#
# ------------------------------------------------------------
# [수정 사항]
#
# - notebook/ipykernel의 -f 인자 무시: parse_known_args()
# - AMP(mixed precision) + GradScaler (CUDA에서만)
# - torch.compile 옵션 반영
# - pin_memory 관련 numpy -> torch 변환 수정
# - full checkpoint 저장/로드 추가
# - --resume 옵션 추가
# - compiled model도 안전하게 state_dict 저장/로드 가능하도록 보완
#
# ============================================================

import os
import time
import json
import random
import argparse
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import gymnasium as gym

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Bernoulli

# Excel
try:
    from openpyxl import Workbook
    from openpyxl.utils import get_column_letter
    OPENPYXL_OK = True
except Exception:
    OPENPYXL_OK = False

# TensorBoard
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_OK = True
except Exception:
    TENSORBOARD_OK = False


# =========================
# Base hyperparameters
# =========================
GAMMA = 0.99

ACTOR_HIDDEN = 256
CRITIC_HIDDEN = 256
LSTM_LAYERS = 1
LSTM_DROPOUT = 0.1

LR_ACTOR = 2e-4
LR_CRITIC = 2e-4
CLIP_NORM = 0.5

USE_HUBER_VALUE_LOSS = True

# Catch & Climb
BASELINE_BETA = 0.90
EPS = 1e-6

# Aux weights (base)
AUX_KL_COEF = 0.50
AUX_KL_MAX = 5.0

SIL_COEF = 0.50
SIL_MAX = 5.0

# Extra updates sweep candidates (FULL에서 사용)
EXTRA_UPDATES_SWEEP = [0, 1, 2, 4, 6]
EXTRA_UPDATES_CAP = 6

# Event deadzone: 너무 자잘한 변동은 이벤트로 치지 않음
EVENT_MIN_INT = 0.04

# Anchor EMA update
ANCHOR_EMA_BETA = 0.995

# Phase schedule (탐색 -> 수렴)
SCHEDULE_ON = True
SCHEDULE_START_FRAC = 0.60
ENT_COEF_START = 0.010
ENT_COEF_END = 0.003
SIL_SCALE_END = 0.50
KL_SCALE_END = 1.50

# Env / solve criterion
ENV_ID = "CartPole-v1"
MAX_STEPS_PER_EP = 500
SOLVE_AVG100 = 475.0


# =========================
# Repro / device
# =========================
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# =========================
# Tensor helpers
# =========================
def numpy_to_torch(
    arr: np.ndarray,
    dtype: torch.dtype,
    device: torch.device,
    pin_memory: bool = False,
    unsqueeze0: bool = False,
    view_shape: Optional[Tuple[int, ...]] = None,
) -> torch.Tensor:
    """
    numpy -> torch 변환 헬퍼

    주의:
    - torch.tensor(np_arr, pin_memory=True)는 에러 날 수 있음
    - 안전하게 torch.from_numpy(...).pin_memory() 사용
    """
    if not isinstance(arr, np.ndarray):
        arr = np.asarray(arr)

    if not arr.flags["C_CONTIGUOUS"]:
        arr = np.ascontiguousarray(arr)

    t = torch.from_numpy(arr)

    if t.dtype != dtype:
        t = t.to(dtype=dtype)

    if pin_memory and device.type == "cuda":
        t = t.pin_memory()

    if unsqueeze0:
        t = t.unsqueeze(0)

    if view_shape is not None:
        t = t.view(*view_shape)

    return t.to(device, non_blocking=(pin_memory and device.type == "cuda"))


# =========================
# Networks
# =========================
class PolicyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 64)
        self.lstm = nn.LSTM(
            64,
            ACTOR_HIDDEN,
            num_layers=LSTM_LAYERS,
            dropout=(LSTM_DROPOUT if LSTM_LAYERS >= 2 else 0.0),
            batch_first=True,
        )
        self.fc2 = nn.Linear(ACTOR_HIDDEN, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, hidden):
        x = self.relu(self.fc1(x))
        x, hidden = self.lstm(x, hidden)
        x = self.relu(x)
        x = self.sigmoid(self.fc2(x))  # (B,T,1)
        return x, hidden

    @torch.no_grad()
    def select_action(self, state, hidden):
        prob, hidden = self.forward(state, hidden)
        b = Bernoulli(prob)
        action = b.sample()
        return int(action.item()), hidden


class ValueNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 64)
        self.lstm = nn.LSTM(
            64,
            CRITIC_HIDDEN,
            num_layers=LSTM_LAYERS,
            dropout=(LSTM_DROPOUT if LSTM_LAYERS >= 2 else 0.0),
            batch_first=True,
        )
        self.fc2 = nn.Linear(CRITIC_HIDDEN, 1)
        self.relu = nn.ReLU()

    def forward(self, x, hidden):
        x = self.relu(self.fc1(x))
        x, hidden = self.lstm(x, hidden)
        x = self.relu(x)
        x = self.fc2(x)
        return x, hidden


# =========================
# Helpers
# =========================
def unwrap_model(model: nn.Module) -> nn.Module:
    """
    torch.compile(model)된 경우 state_dict key가 꼬이지 않게
    원본 모듈(_orig_mod)이 있으면 그걸 사용.
    """
    return model._orig_mod if hasattr(model, "_orig_mod") else model


def model_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return unwrap_model(model).state_dict()


def load_model_state_dict(model: nn.Module, state_dict: Dict[str, torch.Tensor], strict: bool = True):
    unwrap_model(model).load_state_dict(state_dict, strict=strict)


def obs_to_partial(obs):
    return np.array([obs[0], obs[2]], dtype=np.float32)


def bernoulli_kl(p, q, eps=1e-6):
    """
    KL(Bern(p) || Bern(q))
    """
    p = torch.clamp(p, eps, 1.0 - eps)
    q = torch.clamp(q, eps, 1.0 - eps)
    return p * torch.log(p / q) + (1.0 - p) * torch.log((1.0 - p) / (1.0 - q))


@torch.no_grad()
def forward_policy_probs(policy_net, states_tensor, device):
    a_hx = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)
    a_cx = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)
    prob, _ = policy_net(states_tensor, (a_hx, a_cx))
    return prob.squeeze(0)  # (T,1)


def compute_climb_signals(ep_reward, baseline_prev):
    denom = abs(baseline_prev) + 1.0
    collapse_raw = max(0.0, baseline_prev - ep_reward)
    spike_raw = max(0.0, ep_reward - baseline_prev)

    collapse_int = collapse_raw / (denom + EPS)
    spike_int = spike_raw / (denom + EPS)

    is_collapse = 1.0 if (ep_reward < baseline_prev and collapse_int >= EVENT_MIN_INT) else 0.0
    is_spike = 1.0 if (ep_reward > baseline_prev and spike_int >= EVENT_MIN_INT) else 0.0
    return is_collapse, is_spike, collapse_int, spike_int


def discounted_returns(rewards: List[float], gamma: float) -> np.ndarray:
    out = np.zeros(len(rewards), dtype=np.float32)
    R = 0.0
    for i in reversed(range(len(rewards))):
        R = gamma * R + rewards[i]
        out[i] = R
    return out


def compute_metrics(reward_list: List[float], solve_thr_avg100: float = SOLVE_AVG100) -> Dict[str, float]:
    r = np.asarray(reward_list, dtype=np.float32)
    auc_mean = float(r.mean())

    tail = r[-100:] if len(r) >= 100 else r
    last100_mean = float(tail.mean())
    last100_std = float(tail.std(ddof=0))

    first_solve = -1
    if len(r) >= 100:
        win = np.convolve(r, np.ones(100, dtype=np.float32) / 100.0, mode="valid")
        idx = np.where(win >= solve_thr_avg100)[0]
        if len(idx) > 0:
            first_solve = int(idx[0] + 100)

    eps = 1e-6
    stability = 1.0 - (last100_std / (abs(last100_mean) + eps))
    stability = float(max(-5.0, min(1.0, stability)))

    return {
        "auc_mean": auc_mean,
        "last100_mean": last100_mean,
        "last100_std": last100_std,
        "first_solve_ep(avg100>=thr)": float(first_solve),
        "stability_last100": stability,
    }


def linear_interp(a: float, b: float, t01: float) -> float:
    t01 = float(max(0.0, min(1.0, t01)))
    return a + (b - a) * t01


def phase_schedule(epoch: int, episodes: int) -> Dict[str, float]:
    """
    탐색 -> 수렴 스케줄
    """
    if (not SCHEDULE_ON) or episodes <= 1:
        return {"ent_coef": ENT_COEF_START, "sil_scale": 1.0, "kl_scale": 1.0}

    frac = epoch / float(episodes - 1)
    if frac <= SCHEDULE_START_FRAC:
        return {"ent_coef": ENT_COEF_START, "sil_scale": 1.0, "kl_scale": 1.0}

    t = (frac - SCHEDULE_START_FRAC) / (1.0 - SCHEDULE_START_FRAC)
    ent = linear_interp(ENT_COEF_START, ENT_COEF_END, t)
    sil_scale = linear_interp(1.0, SIL_SCALE_END, t)
    kl_scale = linear_interp(1.0, KL_SCALE_END, t)
    return {"ent_coef": ent, "sil_scale": sil_scale, "kl_scale": kl_scale}


# =========================
# Ablation config
# =========================
@dataclass
class Variant:
    name: str
    use_kl_anchor: bool = True
    use_sil: bool = True
    extra_on_collapse: bool = True
    extra_on_spike: bool = True
    use_entropy: bool = True
    use_adv_norm: bool = True


def get_variants() -> List[Variant]:
    return [
        Variant("FULL",       use_kl_anchor=True,  use_sil=True,  extra_on_collapse=True,  extra_on_spike=True,  use_entropy=True, use_adv_norm=True),
        Variant("NoSpike",    use_kl_anchor=True,  use_sil=False, extra_on_collapse=True,  extra_on_spike=False, use_entropy=True, use_adv_norm=True),
        Variant("NoEventUpd", use_kl_anchor=True,  use_sil=True,  extra_on_collapse=False, extra_on_spike=False, use_entropy=True, use_adv_norm=True),
        Variant("NoCollapse", use_kl_anchor=False, use_sil=True,  extra_on_collapse=False, extra_on_spike=True,  use_entropy=True, use_adv_norm=True),
        Variant("Vanilla",    use_kl_anchor=False, use_sil=False, extra_on_collapse=False, extra_on_spike=False, use_entropy=True, use_adv_norm=True),
    ]


# =========================
# Checkpoint utils
# =========================
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def get_ckpt_path(ckpt_dir: str, variant_name: str, eu: int, seed: int) -> str:
    ensure_dir(ckpt_dir)
    return os.path.join(ckpt_dir, f"{variant_name}_EU{eu}_seed{seed}_latest.pt")


def get_meta_path(ckpt_dir: str, variant_name: str, eu: int, seed: int) -> str:
    ensure_dir(ckpt_dir)
    return os.path.join(ckpt_dir, f"{variant_name}_EU{eu}_seed{seed}_meta.json")


def save_json(path: str, data: Dict[str, object]):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json(path: str) -> Dict[str, object]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_checkpoint(
    path: str,
    epoch: int,
    policy: nn.Module,
    value: nn.Module,
    optim: torch.optim.Optimizer,
    value_optim: torch.optim.Optimizer,
    anchor_policy: nn.Module,
    baseline: float,
    ep_rewards: List[float],
    collapse_count: int,
    spike_count: int,
):
    ckpt = {
        "epoch": epoch,
        "policy": model_state_dict(policy),
        "value": model_state_dict(value),
        "optim": optim.state_dict(),
        "value_optim": value_optim.state_dict(),
        "anchor_policy": model_state_dict(anchor_policy),
        "baseline": baseline,
        "ep_rewards": ep_rewards,
        "collapse_count": collapse_count,
        "spike_count": spike_count,
    }
    torch.save(ckpt, path)


def load_checkpoint(
    path: str,
    policy: nn.Module,
    value: nn.Module,
    optim: torch.optim.Optimizer,
    value_optim: torch.optim.Optimizer,
    anchor_policy: nn.Module,
    device: torch.device,
):
    ckpt = torch.load(path, map_location=device)

    load_model_state_dict(policy, ckpt["policy"], strict=True)
    load_model_state_dict(value, ckpt["value"], strict=True)
    optim.load_state_dict(ckpt["optim"])
    value_optim.load_state_dict(ckpt["value_optim"])
    load_model_state_dict(anchor_policy, ckpt["anchor_policy"], strict=True)

    start_epoch = int(ckpt["epoch"]) + 1
    baseline = float(ckpt.get("baseline", 0.0))
    ep_rewards = list(ckpt.get("ep_rewards", []))
    collapse_count = int(ckpt.get("collapse_count", 0))
    spike_count = int(ckpt.get("spike_count", 0))

    return start_epoch, baseline, ep_rewards, collapse_count, spike_count


# =========================
# One run (one seed, one variant)
# =========================
def run_variant(
    variant: Variant,
    episodes: int,
    seed: int,
    extra_updates_on_event: int,
    log_dir_root: Optional[str],
    device: torch.device,
    amp: bool = True,
    save_policy_every: int = 0,
    verbose_every: int = 200,
    compile_models: bool = False,
    ckpt_dir: str = "./checkpoints",
    save_every: int = 200,
    resume: bool = False,
) -> Dict[str, object]:
    set_global_seed(seed)
    env = gym.make(ENV_ID)

    policy = PolicyNetwork().to(device)
    value = ValueNetwork().to(device)

    if compile_models:
        policy = maybe_compile(policy, True)
        value = maybe_compile(value, True)

    optim = torch.optim.Adam(unwrap_model(policy).parameters(), lr=LR_ACTOR)
    value_optim = torch.optim.Adam(unwrap_model(value).parameters(), lr=LR_CRITIC)

    anchor_policy = PolicyNetwork().to(device)
    load_model_state_dict(anchor_policy, model_state_dict(policy), strict=True)
    anchor_policy.eval()

    if compile_models:
        anchor_policy = maybe_compile(anchor_policy, True)

    use_amp = bool(amp) and (device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    baseline = 0.0
    start_epoch = 0
    ep_rewards: List[float] = []
    collapse_count = 0
    spike_count = 0

    ckpt_path = get_ckpt_path(ckpt_dir, variant.name, extra_updates_on_event, seed)
    meta_path = get_meta_path(ckpt_dir, variant.name, extra_updates_on_event, seed)

    if resume and os.path.exists(ckpt_path):
        print(f"[resume] loading checkpoint: {ckpt_path}")
        start_epoch, baseline, ep_rewards, collapse_count, spike_count = load_checkpoint(
            ckpt_path, policy, value, optim, value_optim, anchor_policy, device
        )
        print(f"[resume] resumed from epoch={start_epoch} / total={episodes}")

    # TensorBoard
    writer = None
    run_tag = f"{variant.name}_EU{extra_updates_on_event}_seed{seed}"
    if log_dir_root is not None and TENSORBOARD_OK:
        ensure_dir(log_dir_root)
        writer = SummaryWriter(os.path.join(log_dir_root, run_tag))

    pin_mem = (device.type == "cuda")

    if start_epoch >= episodes:
        print(f"[skip] already finished: {variant.name} EU={extra_updates_on_event} seed={seed}")
        metrics = compute_metrics(ep_rewards, SOLVE_AVG100)
        out = {
            "variant": variant.name,
            "seed": seed,
            "episodes": episodes,
            "extra_updates_on_event": int(extra_updates_on_event),
            "collapse_count": float(collapse_count),
            "spike_count": float(spike_count),
            **metrics,
        }
        if writer is not None:
            writer.close()
        env.close()
        return out

    for epoch in range(start_epoch, episodes):
        sched = phase_schedule(epoch, episodes)
        ent_coef = sched["ent_coef"]
        sil_scale = sched["sil_scale"]
        kl_scale = sched["kl_scale"]

        obs, _info = env.reset(seed=seed * 100000 + epoch)
        state = obs_to_partial(obs)
        episode_reward = 0.0

        a_hx = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)
        a_cx = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)

        rewards, actions, states = [], [], []

        for _t in range(MAX_STEPS_PER_EP):
            states.append(state.copy())

            state_t = torch.as_tensor(state, dtype=torch.float32, device=device).view(1, 1, 2)
            action, (a_hx, a_cx) = policy.select_action(state_t, (a_hx, a_cx))
            actions.append(action)

            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            state = obs_to_partial(next_obs)
            episode_reward += float(reward)
            rewards.append(float(reward))

            if done:
                break

        ep_rewards.append(float(episode_reward))

        returns = discounted_returns(rewards, GAMMA)
        mean, std = returns.mean(), returns.std()
        std = std if std > 1e-8 else 1.0
        returns_norm = (returns - mean) / std

        states_np = np.asarray(states, dtype=np.float32)
        actions_np = np.asarray(actions, dtype=np.float32)
        returns_np = np.asarray(returns_norm, dtype=np.float32)

        states_tensor = numpy_to_torch(
            states_np, dtype=torch.float32, device=device, pin_memory=pin_mem, unsqueeze0=True
        )
        actions_tensor = numpy_to_torch(
            actions_np, dtype=torch.float32, device=device, pin_memory=pin_mem, view_shape=(-1, 1)
        )
        returns_tensor = numpy_to_torch(
            returns_np, dtype=torch.float32, device=device, pin_memory=pin_mem, view_shape=(-1, 1)
        )

        baseline_prev = float(baseline)
        is_collapse, is_spike, collapse_int, spike_int = compute_climb_signals(episode_reward, baseline_prev)
        baseline = BASELINE_BETA * baseline + (1.0 - BASELINE_BETA) * float(episode_reward)

        if is_collapse:
            collapse_count += 1
        if is_spike:
            spike_count += 1

        kl_w = 0.0
        sil_w = 0.0

        if variant.use_kl_anchor and is_collapse:
            kl_w = min(AUX_KL_MAX, AUX_KL_COEF * (1.0 + 5.0 * collapse_int))
            kl_w *= kl_scale

        if variant.use_sil and is_spike:
            sil_w = min(SIL_MAX, SIL_COEF * (1.0 + 5.0 * spike_int))
            sil_w *= sil_scale

        extra_updates = 0
        if is_collapse and variant.extra_on_collapse:
            extra_updates = max(extra_updates, extra_updates_on_event)
        if is_spike and variant.extra_on_spike:
            extra_updates = max(extra_updates, extra_updates_on_event)
        extra_updates = int(min(EXTRA_UPDATES_CAP, extra_updates))

        with torch.no_grad():
            c_hx = torch.zeros((LSTM_LAYERS, 1, CRITIC_HIDDEN), device=device)
            c_cx = torch.zeros((LSTM_LAYERS, 1, CRITIC_HIDDEN), device=device)
            v, _ = value(states_tensor, (c_hx, c_cx))
            v = v.squeeze(0)
            advantage = returns_tensor - v
            if variant.use_adv_norm:
                advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-8)

        anchor_prob = None
        if variant.use_kl_anchor:
            with torch.no_grad():
                anchor_prob = forward_policy_probs(anchor_policy, states_tensor, device)

        def actor_step() -> Tuple[float, float, float, float, float]:
            a_hx0 = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)
            a_cx0 = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)

            with torch.amp.autocast("cuda", enabled=use_amp):
                prob, _ = policy(states_tensor, (a_hx0, a_cx0))
                prob = prob.squeeze(0)

                dist = Bernoulli(prob)
                log_prob = dist.log_prob(actions_tensor)
                entropy = dist.entropy().mean()

                ent_term = (ent_coef * entropy) if variant.use_entropy else 0.0
                base_loss = -(log_prob * advantage.detach()).mean() - ent_term

                kl_loss = 0.0
                if kl_w > 0.0 and anchor_prob is not None:
                    kl = bernoulli_kl(prob, anchor_prob).mean()
                    kl_loss = float(kl_w) * kl

                sil_loss = 0.0
                if sil_w > 0.0:
                    pos_adv = torch.clamp(advantage.detach(), min=0.0)
                    sil_loss = float(sil_w) * (-(log_prob * pos_adv).mean())

                loss = base_loss + kl_loss + sil_loss

            optim.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(unwrap_model(policy).parameters(), CLIP_NORM)
            scaler.step(optim)
            scaler.update()

            def f(x):
                return float(x.item()) if hasattr(x, "item") else float(x)

            return f(loss), f(base_loss), f(kl_loss), f(sil_loss), f(entropy)

        actor_total, actor_base, actor_kl, actor_sil, entropy_v = actor_step()
        for _ in range(extra_updates):
            actor_total, actor_base, actor_kl, actor_sil, entropy_v = actor_step()

        c_hx = torch.zeros((LSTM_LAYERS, 1, CRITIC_HIDDEN), device=device)
        c_cx = torch.zeros((LSTM_LAYERS, 1, CRITIC_HIDDEN), device=device)

        with torch.amp.autocast("cuda", enabled=use_amp):
            v_pred, _ = value(states_tensor, (c_hx, c_cx))
            v_pred = v_pred.squeeze(0)
            if USE_HUBER_VALUE_LOSS:
                value_loss = F.smooth_l1_loss(v_pred, returns_tensor)
            else:
                value_loss = F.mse_loss(v_pred, returns_tensor)

        value_optim.zero_grad(set_to_none=True)
        scaler.scale(value_loss).backward()
        scaler.unscale_(value_optim)
        torch.nn.utils.clip_grad_norm_(unwrap_model(value).parameters(), CLIP_NORM)
        scaler.step(value_optim)
        scaler.update()

        if variant.use_kl_anchor:
            with torch.no_grad():
                anchor_sd = model_state_dict(anchor_policy)
                policy_sd = model_state_dict(policy)
                for k in anchor_sd.keys():
                    anchor_sd[k].mul_(ANCHOR_EMA_BETA).add_(policy_sd[k], alpha=(1.0 - ANCHOR_EMA_BETA))
                load_model_state_dict(anchor_policy, anchor_sd, strict=True)
            anchor_policy.eval()

        if writer is not None:
            writer.add_scalar("episode_reward", episode_reward, epoch)
            writer.add_scalar("baseline/ema", baseline, epoch)
            writer.add_scalar("catch/is_collapse", is_collapse, epoch)
            writer.add_scalar("catch/is_spike", is_spike, epoch)
            writer.add_scalar("catch/collapse_int", collapse_int, epoch)
            writer.add_scalar("catch/spike_int", spike_int, epoch)
            writer.add_scalar("sched/ent_coef", ent_coef, epoch)
            writer.add_scalar("sched/sil_scale", sil_scale, epoch)
            writer.add_scalar("sched/kl_scale", kl_scale, epoch)
            writer.add_scalar("catch/kl_weight", kl_w, epoch)
            writer.add_scalar("catch/sil_weight", sil_w, epoch)
            writer.add_scalar("catch/extra_updates", extra_updates, epoch)
            writer.add_scalar("loss/actor_total", actor_total, epoch)
            writer.add_scalar("loss/actor_base", actor_base, epoch)
            writer.add_scalar("loss/actor_kl", actor_kl, epoch)
            writer.add_scalar("loss/actor_sil", actor_sil, epoch)
            writer.add_scalar("loss/value", float(value_loss.item()), epoch)
            writer.add_scalar("stats/entropy", entropy_v, epoch)

        if (epoch % verbose_every) == 0:
            tag = "COLLAPSE" if is_collapse else ("SPIKE" if is_spike else "normal")
            print(
                f"[{variant.name} EU={extra_updates_on_event} seed={seed}] "
                f"ep {epoch:05d} | R {episode_reward:6.1f} | ema {baseline:7.2f} | "
                f"{tag} | extra={extra_updates}"
            )

        if save_policy_every and (epoch + 1) % save_policy_every == 0:
            torch.save(
                model_state_dict(policy),
                os.path.join(
                    ckpt_dir,
                    f"policy_{variant.name}_EU{extra_updates_on_event}_seed{seed}_ep{epoch+1}.pt",
                ),
            )

        if save_every and (epoch + 1) % save_every == 0:
            save_checkpoint(
                ckpt_path,
                epoch,
                policy,
                value,
                optim,
                value_optim,
                anchor_policy,
                baseline,
                ep_rewards,
                collapse_count,
                spike_count,
            )
            save_json(meta_path, {
                "variant": variant.name,
                "seed": seed,
                "episodes_total": episodes,
                "last_saved_epoch": epoch,
                "extra_updates_on_event": extra_updates_on_event,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            })
            print(f"[ckpt] saved: {ckpt_path} @ ep {epoch+1}")

    save_checkpoint(
        ckpt_path,
        episodes - 1,
        policy,
        value,
        optim,
        value_optim,
        anchor_policy,
        baseline,
        ep_rewards,
        collapse_count,
        spike_count,
    )
    save_json(meta_path, {
        "variant": variant.name,
        "seed": seed,
        "episodes_total": episodes,
        "last_saved_epoch": episodes - 1,
        "extra_updates_on_event": extra_updates_on_event,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "status": "finished",
    })
    print(f"[ckpt] final saved: {ckpt_path}")

    if writer is not None:
        writer.close()
    env.close()

    metrics = compute_metrics(ep_rewards, SOLVE_AVG100)
    out = {
        "variant": variant.name,
        "seed": seed,
        "episodes": episodes,
        "extra_updates_on_event": int(extra_updates_on_event),
        "collapse_count": float(collapse_count),
        "spike_count": float(spike_count),
        **metrics,
    }
    return out


# =========================
# Aggregation
# =========================
def aggregate_across_seeds(results: List[Dict[str, object]]) -> List[Dict[str, object]]:
    by: Dict[str, List[Dict[str, object]]] = {}
    for r in results:
        key = f"{r['variant']}_EU{r.get('extra_updates_on_event', 0)}"
        by.setdefault(key, []).append(r)

    keys = [
        "auc_mean",
        "last100_mean",
        "last100_std",
        "first_solve_ep(avg100>=thr)",
        "stability_last100",
        "collapse_count",
        "spike_count",
    ]

    agg = []
    for kname, rs in by.items():
        out = {"group": kname, "n_seeds": len(rs)}
        out["variant"] = rs[0]["variant"]
        out["extra_updates_on_event"] = int(rs[0].get("extra_updates_on_event", 0))
        for k in keys:
            vals = [float(x[k]) for x in rs]
            out[k] = float(np.mean(vals))
        agg.append(out)

    agg.sort(key=lambda x: x["last100_mean"], reverse=True)
    return agg


def aggregate_by_variant_only(results: List[Dict[str, object]]) -> List[Dict[str, object]]:
    by: Dict[str, List[Dict[str, object]]] = {}
    for r in results:
        by.setdefault(str(r["variant"]), []).append(r)

    keys = [
        "auc_mean",
        "last100_mean",
        "last100_std",
        "first_solve_ep(avg100>=thr)",
        "stability_last100",
        "collapse_count",
        "spike_count",
    ]

    agg = []
    for v, rs in by.items():
        out = {"variant": v, "n_seeds": len(rs)}
        out["extra_updates_on_event"] = int(rs[0].get("extra_updates_on_event", 0))
        for k in keys:
            vals = [float(x[k]) for x in rs]
            out[k] = float(np.mean(vals))
        agg.append(out)

    agg.sort(key=lambda x: x["last100_mean"], reverse=True)
    return agg


# =========================
# Excel writer
# =========================
def autosize_worksheet(ws):
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                v = "" if cell.value is None else str(cell.value)
                max_len = max(max_len, len(v))
            except Exception:
                pass
        ws.column_dimensions[col_letter].width = min(60, max(10, max_len + 2))


def write_sheet(ws, rows: List[Dict[str, object]], title: str = ""):
    if title:
        ws.append([title])
        ws.append([])

    if not rows:
        ws.append(["(no rows)"])
        return

    cols = sorted({k for r in rows for k in r.keys()})
    ws.append(cols)
    for r in rows:
        ws.append([r.get(c, "") for c in cols])

    autosize_worksheet(ws)


def save_to_excel(
    path: str,
    sweep_raw: List[Dict[str, object]],
    sweep_summary: List[Dict[str, object]],
    ablation_raw: List[Dict[str, object]],
    ablation_summary: List[Dict[str, object]],
):
    if not OPENPYXL_OK:
        raise RuntimeError("openpyxl is not available. Please `pip install openpyxl`.")

    wb = Workbook()
    ws0 = wb.active
    ws0.title = "readme"
    ws0.append(["test_6 results workbook"])
    ws0.append(["Sheets: sweep_raw, sweep_summary, ablation_raw, ablation_summary"])
    ws0.append(["Saved at:", time.strftime("%Y-%m-%d %H:%M:%S")])

    ws = wb.create_sheet("sweep_raw")
    write_sheet(ws, sweep_raw, "FULL extra_updates sweep (raw per seed)")

    ws = wb.create_sheet("sweep_summary")
    write_sheet(ws, sweep_summary, "FULL extra_updates sweep (summary across seeds)")

    ws = wb.create_sheet("ablation_raw")
    write_sheet(ws, ablation_raw, "Ablation (raw per seed)")

    ws = wb.create_sheet("ablation_summary")
    write_sheet(ws, ablation_summary, "Ablation (summary across seeds)")

    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    wb.save(path)


# =========================
# Selection logic for best EU
# =========================
def pick_best_extra_updates(sweep_summary: List[Dict[str, object]]) -> int:
    """
    우선순위:
    (1) solve 여부
    (2) AUC
    (3) last100_mean
    (4) stability
    """
    def key_fn(r):
        first = float(r.get("first_solve_ep(avg100>=thr)", -1))
        solved = 1 if first > 0 else 0
        return (
            solved,
            float(r.get("auc_mean", 0.0)),
            float(r.get("last100_mean", 0.0)),
            float(r.get("stability_last100", -999.0)),
            -first if solved else 0.0,
        )

    best = max(sweep_summary, key=key_fn)
    return int(best["extra_updates_on_event"])


# =========================
# Args / compile
# =========================
def parse_args():
    p = argparse.ArgumentParser(add_help=True)

    p.add_argument("--device", type=str, default=None, help="e.g. cuda:0 / cuda:1 / cpu. default: auto")
    p.add_argument("--amp", action="store_true", help="use mixed precision on cuda")
    p.add_argument("--no-amp", dest="amp", action="store_false")
    p.set_defaults(amp=True)

    p.add_argument("--compile", action="store_true", help="torch.compile models (PyTorch 2.x)")
    p.add_argument("--logdir", type=str, default="./runs_ablation", help="TensorBoard root (set '' to disable)")
    p.add_argument("--xlsx", type=str, default="./test_6.xlsx", help="output xlsx path")

    p.add_argument("--save-policy-every", type=int, default=0, help="save policy-only weights every N episodes")
    p.add_argument("--verbose-every", type=int, default=200)

    p.add_argument("--episodes-sweep", type=int, default=5000)
    p.add_argument("--episodes-ablation", type=int, default=5000)
    p.add_argument("--seeds", type=str, default="0,1,2,3,4", help="comma-separated seeds")

    # checkpoint / resume
    p.add_argument("--ckpt-dir", type=str, default="./checkpoints", help="checkpoint directory")
    p.add_argument("--save-every", type=int, default=200, help="save full checkpoint every N episodes")
    p.add_argument("--resume", action="store_true", help="resume from latest checkpoint if exists")

    # Jupyter(ipykernel)에서 자동으로 붙는 -f 같은 인자 무시
    args, _unknown = p.parse_known_args()
    return args


def maybe_compile(model: nn.Module, enabled: bool) -> nn.Module:
    if not enabled:
        return model
    try:
        return torch.compile(model)
    except Exception as e:
        print("[warn] torch.compile failed -> fallback. err:", repr(e))
        return model


# =========================
# Main
# =========================
def main():
    args = parse_args()

    if args.device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(args.device)

    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    print("device =", device, "| amp =", (args.amp and device.type == "cuda"), "| compile =", args.compile)

    SEEDS = [int(x.strip()) for x in args.seeds.split(",") if x.strip() != ""]
    EPISODES_SWEEP = int(args.episodes_sweep)
    EPISODES_ABLATION = int(args.episodes_ablation)

    LOG_ROOT = args.logdir if (args.logdir is not None and args.logdir.strip() != "") else None
    SAVE_POLICY_EVERY = int(args.save_policy_every)
    VERBOSE_EVERY = int(args.verbose_every)

    # ---------- (1) FULL: extra_updates sweep ----------
    sweep_raw: List[Dict[str, object]] = []
    full = Variant("FULL", use_kl_anchor=True, use_sil=True, extra_on_collapse=True, extra_on_spike=True)

    print("\n\n===== [1] FULL extra_updates sweep =====")
    for eu in EXTRA_UPDATES_SWEEP:
        for s in SEEDS:
            print(f"\n--- SWEEP RUN: FULL EU={eu} seed={s} episodes={EPISODES_SWEEP} ---")
            res = run_variant(
                variant=full,
                episodes=EPISODES_SWEEP,
                seed=s,
                extra_updates_on_event=eu,
                log_dir_root=LOG_ROOT,
                device=device,
                amp=args.amp,
                save_policy_every=SAVE_POLICY_EVERY,
                verbose_every=VERBOSE_EVERY,
                compile_models=args.compile,
                ckpt_dir=args.ckpt_dir,
                save_every=args.save_every,
                resume=args.resume,
            )
            sweep_raw.append(res)

    sweep_summary = aggregate_across_seeds(sweep_raw)

    best_eu = pick_best_extra_updates(sweep_summary)
    print("\n===== Sweep summary (top 5 by last100_mean) =====")
    for r in sweep_summary[:5]:
        print(r)
    print(f"\n>>> Picked best extra_updates_on_event for FINAL ABLATION: {best_eu}")

    # ---------- (2) Final Ablation with best EU ----------
    print("\n\n===== [2] FINAL ABLATION (best EU) =====")
    variants = get_variants()
    ablation_raw: List[Dict[str, object]] = []
    for v in variants:
        for s in SEEDS:
            print(f"\n=== ABLATION RUN: {v.name} EU={best_eu} seed={s} episodes={EPISODES_ABLATION} ===")
            res = run_variant(
                variant=v,
                episodes=EPISODES_ABLATION,
                seed=s,
                extra_updates_on_event=best_eu,
                log_dir_root=LOG_ROOT,
                device=device,
                amp=args.amp,
                save_policy_every=SAVE_POLICY_EVERY,
                verbose_every=VERBOSE_EVERY,
                compile_models=args.compile,
                ckpt_dir=args.ckpt_dir,
                save_every=args.save_every,
                resume=args.resume,
            )
            ablation_raw.append(res)

    ablation_summary = aggregate_by_variant_only(ablation_raw)

    # ---------- Save Excel ----------
    out_xlsx = args.xlsx
    save_to_excel(out_xlsx, sweep_raw, sweep_summary, ablation_raw, ablation_summary)
    print(f"\nSaved Excel: {out_xlsx}")

    # ---------- Print quick table ----------
    print("\n\n========== FINAL ABLATION SUMMARY (mean across seeds) ==========")
    cols = [
        "variant", "n_seeds",
        "auc_mean", "last100_mean", "last100_std",
        "first_solve_ep(avg100>=thr)",
        "stability_last100",
        "collapse_count", "spike_count",
        "extra_updates_on_event",
    ]

    def fmt_row(r):
        out = {}
        for c in cols:
            out[c] = r.get(c, "")
        for k in ["auc_mean", "last100_mean", "last100_std", "stability_last100"]:
            if k in out and out[k] != "":
                out[k] = f"{float(out[k]):.3f}"
        if "first_solve_ep(avg100>=thr)" in out and out["first_solve_ep(avg100>=thr)"] != "":
            out["first_solve_ep(avg100>=thr)"] = int(float(out["first_solve_ep(avg100>=thr)"]))
        for k in ["collapse_count", "spike_count"]:
            if k in out and out[k] != "":
                out[k] = f"{float(out[k]):.1f}"
        return out

    pretty = [fmt_row(r) for r in ablation_summary]
    widths = {c: max(len(c), max(len(str(pr.get(c, ""))) for pr in pretty)) for c in cols}
    header = " | ".join(c.ljust(widths[c]) for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for pr in pretty:
        print(" | ".join(str(pr.get(c, "")).ljust(widths[c]) for c in cols))

    if LOG_ROOT is not None and TENSORBOARD_OK:
        print(f"\nTensorBoard: tensorboard --logdir {LOG_ROOT}")

    print("\n[server run example]")
    print(
        "nohup python ablate_runner.py "
        "--device cuda:0 --resume "
        "--save-every 200 --save-policy-every 200 "
        "--ckpt-dir ./checkpoints "
        "--logdir ./runs_ablation "
        "--xlsx ./test_6.xlsx "
        "> train.log 2>&1 &"
    )


if __name__ == "__main__":
    main()

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


device = cuda:0 | amp = True | compile = False


===== [1] FULL extra_updates sweep =====

--- SWEEP RUN: FULL EU=0 seed=0 episodes=5000 ---
[FULL EU=0 seed=0] ep 00000 | R   29.0 | ema    2.90 | SPIKE | extra=0
[ckpt] saved: ./checkpoints/FULL_EU0_seed0_latest.pt @ ep 200
[FULL EU=0 seed=0] ep 00200 | R   14.0 | ema   25.32 | COLLAPSE | extra=0
[ckpt] saved: ./checkpoints/FULL_EU0_seed0_latest.pt @ ep 400
[FULL EU=0 seed=0] ep 00400 | R   33.0 | ema   24.11 | SPIKE | extra=0
[ckpt] saved: ./checkpoints/FULL_EU0_seed0_latest.pt @ ep 600
[FULL EU=0 seed=0] ep 00600 | R   28.0 | ema   25.81 | SPIKE | extra=0
[ckpt] saved: ./checkpoints/FULL_EU0_seed0_latest.pt @ ep 800
[FULL EU=0 seed=0] ep 00800 | R   15.0 | ema   26.13 | COLLAPSE | extra=0
[ckpt] saved: ./checkpoints/FULL_EU0_seed0_latest.pt @ ep 1000
[FULL EU=0 seed=0] ep 01000 | R   27.0 | ema   34.87 | COLLAPSE | extra=0
[ckpt] saved: ./checkpoints/FULL_EU0_seed0_latest.pt @ ep 1200
[FULL EU=0 seed=0] ep 01200 | R   46.0 | ema   44.

In [ ]:
# pomdp_multi_env_runner.py
# ============================================================
# Multi-Environment POMDP Recurrent PPO Runner
#
# 목적
# - CartPole 전용 실험 코드를 여러 discrete-control 환경으로 확장
# - POMDP(부분관측) 설정에서도 동작하도록 recurrent(LSTM) 정책 사용
# - 학습 / 평가 / 체크포인트 / resume / graceful shutdown / GPU 정리 포함
#
# 지원 환경 예시
# - CartPole-v1
# - Acrobot-v1
# - MountainCar-v0
#
# 핵심 아이디어
# 1) 환경별로 observation 일부만 보게 만들 수 있음 (partial indices)
# 2) flicker / noise를 추가해 POMDP 난이도 조절
# 3) LSTM actor-critic + PPO로 학습
# 4) 여러 preset / seed를 자동 반복
# 5) 종료 시 checkpoint 저장 + CUDA cache 정리
#
# 실행 예시
# python pomdp_multi_env_runner.py --device cuda:0 \
#   --presets cartpole_pomdp,acrobot_pomdp,mountaincar_pomdp \
#   --seeds 0,1,2 --total-episodes 1500
#
# resume
# python pomdp_multi_env_runner.py --device cuda:0 --resume
#
# TensorBoard
# tensorboard --logdir ./runs_pomdp
# ============================================================

import os
import gc
import csv
import time
import json
import math
import signal
import random
import argparse
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import gymnasium as gym

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

# Excel
try:
    from openpyxl import Workbook
    from openpyxl.utils import get_column_letter
    OPENPYXL_OK = True
except Exception:
    OPENPYXL_OK = False

# TensorBoard
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_OK = True
except Exception:
    TENSORBOARD_OK = False


# ============================================================
# Global constants
# ============================================================
DEFAULT_TOTAL_EPISODES = 1500
DEFAULT_MAX_STEPS_PER_EPISODE = 500
DEFAULT_STEPS_PER_UPDATE = 1024
DEFAULT_EVAL_EPISODES = 10
DEFAULT_SAVE_EVERY = 100
DEFAULT_EVAL_EVERY = 100
DEFAULT_LOG_EVERY = 20

GAMMA = 0.99
GAE_LAMBDA = 0.95
PPO_CLIP = 0.2
PPO_EPOCHS = 4
MINIBATCH_SIZE = 256
LR = 3e-4
ENTROPY_COEF_START = 0.01
ENTROPY_COEF_END = 0.001
VALUE_COEF = 0.5
MAX_GRAD_NORM = 0.5

LSTM_HIDDEN = 128
LSTM_LAYERS = 1
MLP_HIDDEN = 128

USE_AMP_DEFAULT = True

STOP_REQUESTED = False


# ============================================================
# Graceful shutdown
# ============================================================
def _signal_handler(signum, frame):
    global STOP_REQUESTED
    STOP_REQUESTED = True
    print(f"\n[signal] received signal={signum}. will stop after current safe point...")


signal.signal(signal.SIGINT, _signal_handler)
signal.signal(signal.SIGTERM, _signal_handler)


# ============================================================
# Utils
# ============================================================
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def maybe_compile(model: nn.Module, enabled: bool) -> nn.Module:
    if not enabled:
        return model
    try:
        return torch.compile(model)
    except Exception as e:
        print("[warn] torch.compile failed -> fallback:", repr(e))
        return model


def unwrap_model(model: nn.Module) -> nn.Module:
    return model._orig_mod if hasattr(model, "_orig_mod") else model


def model_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return unwrap_model(model).state_dict()


def load_model_state_dict(model: nn.Module, state_dict: Dict[str, torch.Tensor], strict: bool = True):
    unwrap_model(model).load_state_dict(state_dict, strict=strict)


def cleanup_torch(device: torch.device):
    try:
        gc.collect()
        if device.type == "cuda":
            torch.cuda.synchronize(device)
            torch.cuda.empty_cache()
            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass
    except Exception as e:
        print("[warn] cleanup_torch failed:", repr(e))


def linear_interp(a: float, b: float, t01: float) -> float:
    t01 = max(0.0, min(1.0, float(t01)))
    return a + (b - a) * t01


def entropy_schedule(progress01: float) -> float:
    return linear_interp(ENTROPY_COEF_START, ENTROPY_COEF_END, progress01)


# ============================================================
# POMDP wrapper
# ============================================================
class PartialObsWrapper(gym.ObservationWrapper):
    """
    1) 특정 index만 선택 가능
    2) flicker 확률로 observation 전체를 0으로 가릴 수 있음
    3) gaussian noise 추가 가능
    """
    def __init__(
        self,
        env: gym.Env,
        partial_indices: Optional[List[int]] = None,
        flicker_prob: float = 0.0,
        noise_std: float = 0.0,
    ):
        super().__init__(env)
        self.partial_indices = partial_indices
        self.flicker_prob = float(flicker_prob)
        self.noise_std = float(noise_std)

        assert isinstance(env.observation_space, gym.spaces.Box), "Only Box observation supported"
        low = env.observation_space.low
        high = env.observation_space.high

        if self.partial_indices is None:
            obs_dim = int(np.prod(low.shape))
            self.partial_indices = list(range(obs_dim))

        low2 = low[self.partial_indices].astype(np.float32)
        high2 = high[self.partial_indices].astype(np.float32)

        self.observation_space = gym.spaces.Box(
            low=low2,
            high=high2,
            shape=(len(self.partial_indices),),
            dtype=np.float32,
        )

    def observation(self, obs):
        obs = np.asarray(obs, dtype=np.float32)
        obs = obs[self.partial_indices].astype(np.float32)

        if self.flicker_prob > 0.0 and np.random.rand() < self.flicker_prob:
            obs = np.zeros_like(obs, dtype=np.float32)

        if self.noise_std > 0.0:
            obs = obs + np.random.normal(0.0, self.noise_std, size=obs.shape).astype(np.float32)

        return obs.astype(np.float32)


# ============================================================
# Env presets
# ============================================================
@dataclass
class EnvPreset:
    name: str
    env_id: str
    partial_indices: Optional[List[int]]
    flicker_prob: float = 0.0
    noise_std: float = 0.0
    max_steps: int = DEFAULT_MAX_STEPS_PER_EPISODE
    solve_score: Optional[float] = None


def get_presets() -> Dict[str, EnvPreset]:
    return {
        "cartpole_pomdp": EnvPreset(
            name="cartpole_pomdp",
            env_id="CartPole-v1",
            partial_indices=[0, 2],   # 위치, 각도만 사용
            flicker_prob=0.05,
            noise_std=0.01,
            max_steps=500,
            solve_score=475.0,
        ),
        "cartpole_full": EnvPreset(
            name="cartpole_full",
            env_id="CartPole-v1",
            partial_indices=None,
            flicker_prob=0.0,
            noise_std=0.0,
            max_steps=500,
            solve_score=475.0,
        ),
        "acrobot_pomdp": EnvPreset(
            name="acrobot_pomdp",
            env_id="Acrobot-v1",
            partial_indices=[0, 1, 4, 5],  # 일부만
            flicker_prob=0.10,
            noise_std=0.01,
            max_steps=500,
            solve_score=-100.0,  # 높을수록 좋음, 덜 음수일수록 좋음
        ),
        "mountaincar_pomdp": EnvPreset(
            name="mountaincar_pomdp",
            env_id="MountainCar-v0",
            partial_indices=[0],  # 위치만 보여줌
            flicker_prob=0.05,
            noise_std=0.005,
            max_steps=200,
            solve_score=-110.0,
        ),
    }


def build_env(preset: EnvPreset, seed: Optional[int] = None) -> gym.Env:
    env = gym.make(preset.env_id)
    env = PartialObsWrapper(
        env,
        partial_indices=preset.partial_indices,
        flicker_prob=preset.flicker_prob,
        noise_std=preset.noise_std,
    )
    if seed is not None:
        env.reset(seed=seed)
        env.action_space.seed(seed)
    return env


# ============================================================
# Recurrent Actor-Critic
# ============================================================
class RecurrentActorCritic(nn.Module):
    def __init__(self, obs_dim: int, action_dim: int):
        super().__init__()
        self.obs_dim = obs_dim
        self.action_dim = action_dim

        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, MLP_HIDDEN),
            nn.ReLU(),
            nn.Linear(MLP_HIDDEN, MLP_HIDDEN),
            nn.ReLU(),
        )
        self.lstm = nn.LSTM(
            input_size=MLP_HIDDEN,
            hidden_size=LSTM_HIDDEN,
            num_layers=LSTM_LAYERS,
            batch_first=True,
        )
        self.actor = nn.Linear(LSTM_HIDDEN, action_dim)
        self.critic = nn.Linear(LSTM_HIDDEN, 1)

    def forward(
        self,
        obs_seq: torch.Tensor,
        hidden: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    ):
        # obs_seq: (B,T,obs_dim)
        x = self.encoder(obs_seq)
        x, hidden = self.lstm(x, hidden)
        logits = self.actor(x)
        values = self.critic(x)
        return logits, values, hidden

    def initial_state(self, batch_size: int, device: torch.device):
        h = torch.zeros((LSTM_LAYERS, batch_size, LSTM_HIDDEN), device=device)
        c = torch.zeros((LSTM_LAYERS, batch_size, LSTM_HIDDEN), device=device)
        return (h, c)

    @torch.no_grad()
    def act(
        self,
        obs_t: torch.Tensor,
        hidden: Tuple[torch.Tensor, torch.Tensor],
    ):
        # obs_t: (1,1,obs_dim)
        logits, value, hidden = self.forward(obs_t, hidden)
        logits = logits[:, -1, :]  # (1, A)
        value = value[:, -1, :]    # (1, 1)
        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return (
            int(action.item()),
            float(log_prob.item()),
            float(value.item()),
            hidden,
        )


# ============================================================
# Rollout buffer
# ============================================================
class RolloutBuffer:
    def __init__(self):
        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.episode_ids = []

    def clear(self):
        self.__init__()

    def add(self, obs, action, reward, done, log_prob, value, episode_id):
        self.obs.append(np.asarray(obs, dtype=np.float32))
        self.actions.append(int(action))
        self.rewards.append(float(reward))
        self.dones.append(bool(done))
        self.log_probs.append(float(log_prob))
        self.values.append(float(value))
        self.episode_ids.append(int(episode_id))

    def __len__(self):
        return len(self.obs)


# ============================================================
# Return / GAE
# ============================================================
def compute_gae(
    rewards: np.ndarray,
    values: np.ndarray,
    dones: np.ndarray,
    next_value: float,
    gamma: float = GAMMA,
    gae_lambda: float = GAE_LAMBDA,
):
    n = len(rewards)
    advantages = np.zeros(n, dtype=np.float32)
    last_gae = 0.0
    for t in reversed(range(n)):
        if t == n - 1:
            next_nonterminal = 1.0 - float(dones[t])
            next_values = next_value
        else:
            next_nonterminal = 1.0 - float(dones[t])
            next_values = values[t + 1]
        delta = rewards[t] + gamma * next_values * next_nonterminal - values[t]
        last_gae = delta + gamma * gae_lambda * next_nonterminal * last_gae
        advantages[t] = last_gae
    returns = advantages + values
    return advantages, returns


# ============================================================
# Sequence maker for recurrent PPO
# ============================================================
def split_into_episodes(buffer: RolloutBuffer):
    episodes = []
    start = 0
    n = len(buffer)
    if n == 0:
        return episodes

    for i in range(n):
        if buffer.dones[i]:
            end = i + 1
            episodes.append(slice(start, end))
            start = end

    if start < n:
        episodes.append(slice(start, n))
    return episodes


def pad_sequences(arrays: List[np.ndarray], pad_value: float = 0.0):
    max_len = max(a.shape[0] for a in arrays)
    padded = []
    masks = []
    for a in arrays:
        T = a.shape[0]
        pad_shape = (max_len - T,) + a.shape[1:]
        if len(pad_shape) == 1:
            pad_arr = np.full((max_len - T,), pad_value, dtype=a.dtype)
        else:
            pad_arr = np.full(pad_shape, pad_value, dtype=a.dtype)
        padded_arr = np.concatenate([a, pad_arr], axis=0)
        mask = np.concatenate([np.ones(T, dtype=np.float32), np.zeros(max_len - T, dtype=np.float32)], axis=0)
        padded.append(padded_arr)
        masks.append(mask)
    return np.stack(padded), np.stack(masks)


# ============================================================
# Evaluation
# ============================================================
@torch.no_grad()
def evaluate_policy(
    model: nn.Module,
    preset: EnvPreset,
    device: torch.device,
    n_episodes: int = DEFAULT_EVAL_EPISODES,
    greedy: bool = True,
):
    env = build_env(preset, seed=12345)
    scores = []

    try:
        for ep in range(n_episodes):
            obs, _ = env.reset(seed=12345 + ep)
            hidden = unwrap_model(model).initial_state(batch_size=1, device=device)
            done = False
            ep_reward = 0.0
            steps = 0

            while not done and steps < preset.max_steps:
                obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).view(1, 1, -1)
                logits, value, hidden = model(obs_t, hidden)
                logits = logits[:, -1, :]
                dist = Categorical(logits=logits)
                action = torch.argmax(dist.probs, dim=-1) if greedy else dist.sample()
                obs, reward, terminated, truncated, _ = env.step(int(action.item()))
                done = terminated or truncated
                ep_reward += float(reward)
                steps += 1

            scores.append(ep_reward)
    finally:
        env.close()

    arr = np.asarray(scores, dtype=np.float32)
    return {
        "eval_mean": float(arr.mean()) if len(arr) else 0.0,
        "eval_std": float(arr.std(ddof=0)) if len(arr) else 0.0,
        "eval_min": float(arr.min()) if len(arr) else 0.0,
        "eval_max": float(arr.max()) if len(arr) else 0.0,
    }


# ============================================================
# Checkpoint
# ============================================================
def ckpt_path(ckpt_dir: str, preset_name: str, seed: int):
    ensure_dir(ckpt_dir)
    return os.path.join(ckpt_dir, f"{preset_name}_seed{seed}_latest.pt")


def meta_path(ckpt_dir: str, preset_name: str, seed: int):
    ensure_dir(ckpt_dir)
    return os.path.join(ckpt_dir, f"{preset_name}_seed{seed}_meta.json")


def save_json(path: str, data: Dict[str, Any]):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def save_checkpoint(
    path: str,
    episode: int,
    global_step: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    train_rewards: List[float],
    eval_history: List[Dict[str, Any]],
    config: Dict[str, Any],
):
    payload = {
        "episode": int(episode),
        "global_step": int(global_step),
        "model": model_state_dict(model),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "train_rewards": list(train_rewards),
        "eval_history": list(eval_history),
        "config": config,
    }
    torch.save(payload, path)


def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    device: torch.device,
):
    payload = torch.load(path, map_location=device)
    load_model_state_dict(model, payload["model"], strict=True)
    optimizer.load_state_dict(payload["optimizer"])
    try:
        scaler.load_state_dict(payload["scaler"])
    except Exception:
        pass

    return {
        "episode": int(payload.get("episode", -1)),
        "global_step": int(payload.get("global_step", 0)),
        "train_rewards": list(payload.get("train_rewards", [])),
        "eval_history": list(payload.get("eval_history", [])),
        "config": payload.get("config", {}),
    }


# ============================================================
# Metrics
# ============================================================
def compute_train_metrics(rewards: List[float], solve_score: Optional[float]):
    r = np.asarray(rewards, dtype=np.float32)
    if len(r) == 0:
        return {
            "auc_mean": 0.0,
            "last100_mean": 0.0,
            "last100_std": 0.0,
            "best_reward": 0.0,
            "first_solve_ep": -1,
        }

    tail = r[-100:] if len(r) >= 100 else r
    last100_mean = float(tail.mean())
    last100_std = float(tail.std(ddof=0))
    best_reward = float(r.max())
    auc_mean = float(r.mean())

    first_solve_ep = -1
    if solve_score is not None and len(r) >= 100:
        win = np.convolve(r, np.ones(100, dtype=np.float32) / 100.0, mode="valid")
        idx = np.where(win >= solve_score)[0]
        if len(idx) > 0:
            first_solve_ep = int(idx[0] + 100)

    return {
        "auc_mean": auc_mean,
        "last100_mean": last100_mean,
        "last100_std": last100_std,
        "best_reward": best_reward,
        "first_solve_ep": int(first_solve_ep),
    }


# ============================================================
# PPO update
# ============================================================
def ppo_update(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    buffer: RolloutBuffer,
    device: torch.device,
    use_amp: bool,
    progress01: float,
):
    episode_slices = split_into_episodes(buffer)
    if not episode_slices:
        return {
            "loss_total": 0.0,
            "loss_policy": 0.0,
            "loss_value": 0.0,
            "entropy": 0.0,
            "approx_kl": 0.0,
        }

    obs_list = []
    act_list = []
    old_logp_list = []
    adv_list = []
    ret_list = []

    for sl in episode_slices:
        rewards = np.asarray(buffer.rewards[sl], dtype=np.float32)
        values = np.asarray(buffer.values[sl], dtype=np.float32)
        dones = np.asarray(buffer.dones[sl], dtype=np.float32)
        next_value = 0.0
        adv, ret = compute_gae(rewards, values, dones, next_value=next_value)

        obs_arr = np.asarray(buffer.obs[sl], dtype=np.float32)
        act_arr = np.asarray(buffer.actions[sl], dtype=np.int64)
        logp_arr = np.asarray(buffer.log_probs[sl], dtype=np.float32)

        obs_list.append(obs_arr)
        act_list.append(act_arr)
        old_logp_list.append(logp_arr)
        adv_list.append(adv.astype(np.float32))
        ret_list.append(ret.astype(np.float32))

    # normalize adv over all valid steps
    all_adv = np.concatenate(adv_list, axis=0)
    adv_mean = all_adv.mean()
    adv_std = all_adv.std() + 1e-8
    adv_list = [((a - adv_mean) / adv_std).astype(np.float32) for a in adv_list]

    obs_pad, mask = pad_sequences(obs_list, pad_value=0.0)
    act_pad, _ = pad_sequences([a[:, None] for a in act_list], pad_value=0)
    old_logp_pad, _ = pad_sequences([a[:, None] for a in old_logp_list], pad_value=0.0)
    adv_pad, _ = pad_sequences([a[:, None] for a in adv_list], pad_value=0.0)
    ret_pad, _ = pad_sequences([a[:, None] for a in ret_list], pad_value=0.0)

    obs_t = torch.as_tensor(obs_pad, dtype=torch.float32, device=device)
    act_t = torch.as_tensor(act_pad.squeeze(-1), dtype=torch.long, device=device)
    old_logp_t = torch.as_tensor(old_logp_pad.squeeze(-1), dtype=torch.float32, device=device)
    adv_t = torch.as_tensor(adv_pad.squeeze(-1), dtype=torch.float32, device=device)
    ret_t = torch.as_tensor(ret_pad.squeeze(-1), dtype=torch.float32, device=device)
    mask_t = torch.as_tensor(mask, dtype=torch.float32, device=device)

    n_seq = obs_t.shape[0]
    idxs = np.arange(n_seq)

    ent_coef = entropy_schedule(progress01)
    total_loss_v = []
    policy_loss_v = []
    value_loss_v = []
    entropy_v = []
    approx_kl_v = []

    for _ in range(PPO_EPOCHS):
        np.random.shuffle(idxs)

        for start in range(0, n_seq, max(1, MINIBATCH_SIZE // max(1, obs_t.shape[1]))):
            batch_idx = idxs[start:start + max(1, MINIBATCH_SIZE // max(1, obs_t.shape[1]))]
            b_obs = obs_t[batch_idx]
            b_act = act_t[batch_idx]
            b_old_logp = old_logp_t[batch_idx]
            b_adv = adv_t[batch_idx]
            b_ret = ret_t[batch_idx]
            b_mask = mask_t[batch_idx]

            hidden = unwrap_model(model).initial_state(batch_size=b_obs.shape[0], device=device)

            with torch.amp.autocast("cuda", enabled=use_amp):
                logits, values, _ = model(b_obs, hidden)
                values = values.squeeze(-1)

                dist = Categorical(logits=logits)
                new_logp = dist.log_prob(b_act)
                entropy = dist.entropy()

                ratio = torch.exp(new_logp - b_old_logp)
                surr1 = ratio * b_adv
                surr2 = torch.clamp(ratio, 1.0 - PPO_CLIP, 1.0 + PPO_CLIP) * b_adv

                policy_loss = -torch.sum(torch.min(surr1, surr2) * b_mask) / (torch.sum(b_mask) + 1e-8)
                value_loss = torch.sum(F.mse_loss(values, b_ret, reduction="none") * b_mask) / (torch.sum(b_mask) + 1e-8)
                entropy_mean = torch.sum(entropy * b_mask) / (torch.sum(b_mask) + 1e-8)

                loss = policy_loss + VALUE_COEF * value_loss - ent_coef * entropy_mean

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(unwrap_model(model).parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                approx_kl = torch.sum((b_old_logp - new_logp) * b_mask) / (torch.sum(b_mask) + 1e-8)

            total_loss_v.append(float(loss.item()))
            policy_loss_v.append(float(policy_loss.item()))
            value_loss_v.append(float(value_loss.item()))
            entropy_v.append(float(entropy_mean.item()))
            approx_kl_v.append(float(approx_kl.item()))

    return {
        "loss_total": float(np.mean(total_loss_v)) if total_loss_v else 0.0,
        "loss_policy": float(np.mean(policy_loss_v)) if policy_loss_v else 0.0,
        "loss_value": float(np.mean(value_loss_v)) if value_loss_v else 0.0,
        "entropy": float(np.mean(entropy_v)) if entropy_v else 0.0,
        "approx_kl": float(np.mean(approx_kl_v)) if approx_kl_v else 0.0,
    }


# ============================================================
# One experiment run
# ============================================================
def run_experiment(
    preset: EnvPreset,
    seed: int,
    total_episodes: int,
    steps_per_update: int,
    eval_every: int,
    eval_episodes: int,
    save_every: int,
    log_every: int,
    device: torch.device,
    log_root: Optional[str],
    ckpt_dir: str,
    compile_models: bool,
    resume: bool,
    amp: bool,
):
    set_global_seed(seed)

    env = build_env(preset, seed=seed)
    obs_dim = int(np.prod(env.observation_space.shape))
    assert isinstance(env.action_space, gym.spaces.Discrete), "This runner supports discrete action env only"
    action_dim = int(env.action_space.n)

    model = RecurrentActorCritic(obs_dim=obs_dim, action_dim=action_dim).to(device)
    if compile_models:
        model = maybe_compile(model, True)

    optimizer = torch.optim.Adam(unwrap_model(model).parameters(), lr=LR)
    use_amp = bool(amp) and (device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    writer = None
    if log_root is not None and TENSORBOARD_OK:
        ensure_dir(log_root)
        writer = SummaryWriter(os.path.join(log_root, f"{preset.name}_seed{seed}"))

    start_episode = 0
    global_step = 0
    train_rewards: List[float] = []
    eval_history: List[Dict[str, Any]] = []

    cpath = ckpt_path(ckpt_dir, preset.name, seed)
    mpath = meta_path(ckpt_dir, preset.name, seed)

    if resume and os.path.exists(cpath):
        print(f"[resume] loading {cpath}")
        state = load_checkpoint(cpath, model, optimizer, scaler, device)
        start_episode = int(state["episode"]) + 1
        global_step = int(state["global_step"])
        train_rewards = state["train_rewards"]
        eval_history = state["eval_history"]
        print(f"[resume] preset={preset.name} seed={seed} start_episode={start_episode}")

    buffer = RolloutBuffer()

    try:
        if start_episode >= total_episodes:
            print(f"[skip] already finished: {preset.name} seed={seed}")
            metrics = compute_train_metrics(train_rewards, preset.solve_score)
            result = {
                "preset": preset.name,
                "env_id": preset.env_id,
                "seed": seed,
                "episodes": total_episodes,
                **metrics,
            }
            if eval_history:
                result.update(eval_history[-1])
            return result

        obs, _ = env.reset(seed=seed + 123)
        hidden = unwrap_model(model).initial_state(batch_size=1, device=device)
        ep_reward = 0.0
        ep_steps = 0
        episode_idx = start_episode

        while episode_idx < total_episodes:
            progress01 = episode_idx / max(1, total_episodes - 1)

            obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).view(1, 1, -1)
            action, log_prob, value, hidden = unwrap_model(model).act(obs_t, hidden)

            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            buffer.add(
                obs=obs,
                action=action,
                reward=reward,
                done=done,
                log_prob=log_prob,
                value=value,
                episode_id=episode_idx,
            )

            obs = next_obs
            ep_reward += float(reward)
            ep_steps += 1
            global_step += 1

            if done or ep_steps >= preset.max_steps:
                train_rewards.append(float(ep_reward))

                if writer is not None:
                    writer.add_scalar("train/episode_reward", ep_reward, episode_idx)
                    writer.add_scalar("train/episode_steps", ep_steps, episode_idx)

                if (episode_idx % log_every) == 0:
                    recent = train_rewards[-20:] if len(train_rewards) >= 20 else train_rewards
                    recent_mean = float(np.mean(recent)) if recent else 0.0
                    print(
                        f"[{preset.name} seed={seed}] "
                        f"ep={episode_idx:05d} "
                        f"R={ep_reward:8.3f} "
                        f"recent20={recent_mean:8.3f} "
                        f"global_step={global_step}"
                    )

                if ((episode_idx + 1) % eval_every) == 0 or (episode_idx + 1) == total_episodes:
                    eval_result = evaluate_policy(
                        model=model,
                        preset=preset,
                        device=device,
                        n_episodes=eval_episodes,
                        greedy=True,
                    )
                    eval_result["eval_at_episode"] = int(episode_idx + 1)
                    eval_history.append(eval_result)

                    if writer is not None:
                        writer.add_scalar("eval/mean_reward", eval_result["eval_mean"], episode_idx)
                        writer.add_scalar("eval/std_reward", eval_result["eval_std"], episode_idx)

                    print(
                        f"[eval] {preset.name} seed={seed} "
                        f"ep={episode_idx+1} "
                        f"mean={eval_result['eval_mean']:.3f} "
                        f"std={eval_result['eval_std']:.3f}"
                    )

                if ((episode_idx + 1) % save_every) == 0 or STOP_REQUESTED:
                    save_checkpoint(
                        cpath,
                        episode=episode_idx,
                        global_step=global_step,
                        model=model,
                        optimizer=optimizer,
                        scaler=scaler,
                        train_rewards=train_rewards,
                        eval_history=eval_history,
                        config={
                            "preset": asdict(preset),
                            "seed": seed,
                            "total_episodes": total_episodes,
                            "steps_per_update": steps_per_update,
                        },
                    )
                    save_json(mpath, {
                        "preset": preset.name,
                        "env_id": preset.env_id,
                        "seed": seed,
                        "last_saved_episode": episode_idx,
                        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "global_step": global_step,
                    })
                    print(f"[ckpt] saved {cpath}")

                episode_idx += 1
                if STOP_REQUESTED:
                    print("[stop] graceful stop requested. leaving training loop.")
                    break

                obs, _ = env.reset(seed=seed * 100000 + episode_idx)
                hidden = unwrap_model(model).initial_state(batch_size=1, device=device)
                ep_reward = 0.0
                ep_steps = 0

            if len(buffer) >= steps_per_update or (STOP_REQUESTED and len(buffer) > 0):
                update_stats = ppo_update(
                    model=model,
                    optimizer=optimizer,
                    scaler=scaler,
                    buffer=buffer,
                    device=device,
                    use_amp=use_amp,
                    progress01=progress01,
                )

                if writer is not None:
                    writer.add_scalar("loss/total", update_stats["loss_total"], global_step)
                    writer.add_scalar("loss/policy", update_stats["loss_policy"], global_step)
                    writer.add_scalar("loss/value", update_stats["loss_value"], global_step)
                    writer.add_scalar("stats/entropy", update_stats["entropy"], global_step)
                    writer.add_scalar("stats/approx_kl", update_stats["approx_kl"], global_step)

                buffer.clear()

                if STOP_REQUESTED:
                    print("[stop] update finished after stop request.")
                    break

        # final save
        final_episode = min(max(start_episode, len(train_rewards)) - 1, total_episodes - 1)
        if final_episode >= 0:
            save_checkpoint(
                cpath,
                episode=final_episode,
                global_step=global_step,
                model=model,
                optimizer=optimizer,
                scaler=scaler,
                train_rewards=train_rewards,
                eval_history=eval_history,
                config={
                    "preset": asdict(preset),
                    "seed": seed,
                    "total_episodes": total_episodes,
                    "steps_per_update": steps_per_update,
                },
            )
            save_json(mpath, {
                "preset": preset.name,
                "env_id": preset.env_id,
                "seed": seed,
                "last_saved_episode": final_episode,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                "global_step": global_step,
                "status": "finished_or_stopped",
            })
            print(f"[ckpt] final saved {cpath}")

        metrics = compute_train_metrics(train_rewards, preset.solve_score)
        result = {
            "preset": preset.name,
            "env_id": preset.env_id,
            "seed": seed,
            "episodes_completed": len(train_rewards),
            **metrics,
        }
        if eval_history:
            result.update(eval_history[-1])
        return result

    finally:
        try:
            env.close()
        except Exception:
            pass
        if writer is not None:
            writer.close()
        cleanup_torch(device)


# ============================================================
# Aggregation
# ============================================================
def aggregate_results(results: List[Dict[str, Any]]):
    grouped: Dict[str, List[Dict[str, Any]]] = {}
    for r in results:
        grouped.setdefault(str(r["preset"]), []).append(r)

    keys = [
        "auc_mean",
        "last100_mean",
        "last100_std",
        "best_reward",
        "first_solve_ep",
        "eval_mean",
        "eval_std",
    ]

    out = []
    for preset_name, rows in grouped.items():
        item = {
            "preset": preset_name,
            "n_seeds": len(rows),
        }
        for k in keys:
            vals = [float(x[k]) for x in rows if k in x]
            item[k] = float(np.mean(vals)) if vals else 0.0
        out.append(item)

    out.sort(key=lambda x: x["last100_mean"], reverse=True)
    return out


# ============================================================
# Save CSV / Excel
# ============================================================
def save_csv(path: str, rows: List[Dict[str, Any]]):
    ensure_dir(os.path.dirname(path) or ".")
    if not rows:
        with open(path, "w", newline="", encoding="utf-8") as f:
            f.write("")
        return

    cols = sorted({k for r in rows for k in r.keys()})
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=cols)
        writer.writeheader()
        writer.writerows(rows)


def autosize_worksheet(ws):
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                v = "" if cell.value is None else str(cell.value)
                max_len = max(max_len, len(v))
            except Exception:
                pass
        ws.column_dimensions[col_letter].width = min(60, max(10, max_len + 2))


def write_sheet(ws, rows: List[Dict[str, Any]], title: str = ""):
    if title:
        ws.append([title])
        ws.append([])

    if not rows:
        ws.append(["(no rows)"])
        return

    cols = sorted({k for r in rows for k in r.keys()})
    ws.append(cols)
    for r in rows:
        ws.append([r.get(c, "") for c in cols])

    autosize_worksheet(ws)


def save_excel(path: str, raw_rows: List[Dict[str, Any]], summary_rows: List[Dict[str, Any]]):
    if not OPENPYXL_OK:
        print("[warn] openpyxl not installed, skip excel save.")
        return

    wb = Workbook()
    ws0 = wb.active
    ws0.title = "readme"
    ws0.append(["pomdp multi-env results"])
    ws0.append(["saved_at", time.strftime("%Y-%m-%d %H:%M:%S")])

    ws1 = wb.create_sheet("raw")
    write_sheet(ws1, raw_rows, "raw per seed")

    ws2 = wb.create_sheet("summary")
    write_sheet(ws2, summary_rows, "summary across seeds")

    ensure_dir(os.path.dirname(path) or ".")
    wb.save(path)


# ============================================================
# Args
# ============================================================
def parse_args():
    p = argparse.ArgumentParser(add_help=True)

    p.add_argument("--device", type=str, default=None, help="cuda:0 / cpu / auto(None)")
    p.add_argument("--amp", action="store_true", help="use amp on cuda")
    p.add_argument("--no-amp", dest="amp", action="store_false")
    p.set_defaults(amp=USE_AMP_DEFAULT)

    p.add_argument("--compile", action="store_true", help="use torch.compile")
    p.add_argument("--resume", action="store_true", help="resume from latest ckpt")
    p.add_argument("--presets", type=str, default="cartpole_pomdp,acrobot_pomdp,mountaincar_pomdp")
    p.add_argument("--seeds", type=str, default="0,1,2")

    p.add_argument("--total-episodes", type=int, default=DEFAULT_TOTAL_EPISODES)
    p.add_argument("--steps-per-update", type=int, default=DEFAULT_STEPS_PER_UPDATE)
    p.add_argument("--eval-every", type=int, default=DEFAULT_EVAL_EVERY)
    p.add_argument("--eval-episodes", type=int, default=DEFAULT_EVAL_EPISODES)
    p.add_argument("--save-every", type=int, default=DEFAULT_SAVE_EVERY)
    p.add_argument("--log-every", type=int, default=DEFAULT_LOG_EVERY)

    p.add_argument("--ckpt-dir", type=str, default="./checkpoints_pomdp")
    p.add_argument("--logdir", type=str, default="./runs_pomdp")
    p.add_argument("--csv", type=str, default="./pomdp_results_raw.csv")
    p.add_argument("--summary-csv", type=str, default="./pomdp_results_summary.csv")
    p.add_argument("--xlsx", type=str, default="./pomdp_results.xlsx")

    args, _unknown = p.parse_known_args()
    return args


# ============================================================
# Main
# ============================================================
def main():
    args = parse_args()
    presets_map = get_presets()

    if args.device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(args.device)

    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    preset_names = [x.strip() for x in args.presets.split(",") if x.strip()]
    seeds = [int(x.strip()) for x in args.seeds.split(",") if x.strip()]

    selected_presets = []
    for name in preset_names:
        if name not in presets_map:
            raise ValueError(f"unknown preset: {name}. available={list(presets_map.keys())}")
        selected_presets.append(presets_map[name])

    print("============================================================")
    print("device       =", device)
    print("amp          =", bool(args.amp and device.type == "cuda"))
    print("compile      =", args.compile)
    print("presets      =", [p.name for p in selected_presets])
    print("seeds        =", seeds)
    print("episodes     =", args.total_episodes)
    print("steps/update =", args.steps_per_update)
    print("============================================================")

    raw_results = []

    started = time.time()

    for preset in selected_presets:
        for seed in seeds:
            if STOP_REQUESTED:
                break

            print(f"\n===== RUN preset={preset.name} seed={seed} =====")
            result = run_experiment(
                preset=preset,
                seed=seed,
                total_episodes=int(args.total_episodes),
                steps_per_update=int(args.steps_per_update),
                eval_every=int(args.eval_every),
                eval_episodes=int(args.eval_episodes),
                save_every=int(args.save_every),
                log_every=int(args.log_every),
                device=device,
                log_root=(args.logdir if args.logdir.strip() else None),
                ckpt_dir=args.ckpt_dir,
                compile_models=args.compile,
                resume=args.resume,
                amp=args.amp,
            )
            raw_results.append(result)

        if STOP_REQUESTED:
            break

    summary = aggregate_results(raw_results)

    save_csv(args.csv, raw_results)
    save_csv(args.summary_csv, summary)
    save_excel(args.xlsx, raw_results, summary)

    elapsed = time.time() - started

    print("\n================ FINAL SUMMARY ================")
    if summary:
        cols = ["preset", "n_seeds", "auc_mean", "last100_mean", "best_reward", "first_solve_ep", "eval_mean", "eval_std"]
        widths = {c: max(len(c), max(len(str(r.get(c, ""))) for r in summary)) for c in cols}
        header = " | ".join(c.ljust(widths[c]) for c in cols)
        sep = "-+-".join("-" * widths[c] for c in cols)
        print(header)
        print(sep)
        for r in summary:
            pr = dict(r)
            for k in ["auc_mean", "last100_mean", "best_reward", "eval_mean", "eval_std"]:
                if k in pr:
                    pr[k] = f"{float(pr[k]):.3f}"
            print(" | ".join(str(pr.get(c, "")).ljust(widths[c]) for c in cols))
    else:
        print("(no results)")

    print("\nraw csv     :", args.csv)
    print("summary csv :", args.summary_csv)
    print("xlsx        :", args.xlsx)
    print(f"elapsed_sec : {elapsed:.2f}")

    print("\n[run example]")
    print(
        "nohup python pomdp_multi_env_runner.py "
        "--device cuda:0 "
        "--presets cartpole_pomdp,acrobot_pomdp,mountaincar_pomdp "
        "--seeds 0,1,2 "
        "--total-episodes 1500 "
        "--resume "
        "> train.log 2>&1 &"
    )

    print("\n[normal shutdown note]")
    print(
        "이 코드는 종료 시 env.close(), writer.close(), gc.collect(), "
        "torch.cuda.empty_cache()를 호출해서 GPU 메모리를 정리한다."
    )


if __name__ == "__main__":
    main()

In [1]:
from IPython import get_ipython

ip = get_ipython()
if ip is not None and hasattr(ip, "kernel"):
    ip.kernel.do_shutdown(restart=False)